In [1]:
# IMPORT VÀ SETUP
import cv2
import numpy as np
import pyautogui
import time
import random
from mss import MSS
import matplotlib.pyplot as plt

In [10]:
MATCH_THRESHOLD = 0.8
BTN_DICE = "images/btn_dice.png"
BTN_CHALLENGE = "images/btn_challenge.png"
REWARD_IMG = "images/reward_img.png"
VICTORY_IMG = "images/victory_img.png"
STATE_QUIZ = "images/fun_quiz.png"
STATE_ROLL = "images/btn_roll.png"
STATE_GIFT = "images/gift_img.png"
BTN_GCLOSE = "images/gift_btn_close.png"
BTN_GCONFIRM = "images/gift_btn_okay.png"
STATE_BOSS = "images/btn_challenge_boss.png"
BTN_RETURN = "images/btn_return.png"
BTN_QUIZ_CORNER = "images/btn_quiz_corner.png"
EXTRA_STATE = "images/extra_img.png"

# Dành cho tính năng nâng cao hơn
HIST_1 = "images/hist_1.png"
HIST_2 = "images/hist_2.png"
HIST_3 = "images/hist_3.png"
HIST_NONE = "images/hist_none.png"

FIX_COORDS =  None

In [11]:
# TEST CHỤP MÀN HÌNH VÀ KHỞI ĐỘNG MSS
sct = MSS()

# Chụp màn hình (nếu cậu dùng 1 màn hình thì để monitors[1], 2 màn thì tùy chỉnh)
screenshot_obj = sct.grab(sct.monitors[1])
screenshot_np = np.array(screenshot_obj)

# mss chụp ra hệ màu BGRA, matplotlib cần RGB để hiển thị đúng màu
img_rgb = cv2.cvtColor(screenshot_np, cv2.COLOR_BGRA2RGB)

# Bỏ comment để test ảnh chụp màn hình (tôi thì test đủ rồi)
# plt.figure(figsize=(10, 6))
# plt.imshow(img_rgb)
# plt.title("Những gì Bot đang nhìn thấy")
# plt.axis('off')
# plt.show()

In [12]:
# CÁC HÀM CỐT LÕI
def find_image_coordinates(template_path, screen_image):
    template = cv2.imread(template_path)
    if template is None:
        print(f"LỖI: Không tìm thấy file {template_path}.")
        return None
    
    h, w, _ = template.shape
    screen_gray = cv2.cvtColor(screen_image, cv2.COLOR_BGRA2GRAY)
    template_gray = cv2.cvtColor(template, cv2.COLOR_BGR2GRAY)

    result = cv2.matchTemplate(screen_gray, template_gray, cv2.TM_CCOEFF_NORMED)
    min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(result)
            
    if max_val >= MATCH_THRESHOLD:
        center_x = max_loc[0] + (w // 2)
        center_y = max_loc[1] + (h // 2)
        return center_x, center_y, w, h
    return None

def find_all_image_coordinates(template_path, screen_image, threshold=MATCH_THRESHOLD):
    """
    Tìm TẤT CẢ các vị trí khớp với template trên màn hình.
    Trả về một list chứa các tuple tọa độ: [(x1, y1, w, h), (x2, y2, w, h), ...]
    """
    template = cv2.imread(template_path)
    if template is None:
        print(f"LỖI: Không tìm thấy file {template_path}")
        return []
    
    h, w, _ = template.shape
    screen_gray = cv2.cvtColor(screen_image, cv2.COLOR_BGRA2GRAY)
    template_gray = cv2.cvtColor(template, cv2.COLOR_BGR2GRAY)

    result = cv2.matchTemplate(screen_gray, template_gray, cv2.TM_CCOEFF_NORMED)
    
    # Lấy tất cả tọa độ có độ tin cậy >= threshold
    # np.where trả về (array_y, array_x)
    locations = np.where(result >= threshold)
    
    matches = []
    # Thuật toán Gom cụm (Clustering):
    # OpenCV thường trả về một cụm nhiều pixel kế nhau cho cùng 1 điểm match.
    # Ta cần lọc để mỗi nút chỉ lấy 1 điểm duy nhất.
    for pt_y, pt_x in zip(*locations):
        center_x = pt_x + w // 2
        center_y = pt_y + h // 2
        
        is_new = True
        for mx, my, mw, mh in matches:
            # Nếu điểm mới này nằm quá gần một điểm đã có (cách nhau bé hơn chiều cao nút)
            # thì coi như chúng nó cùng thuộc 1 nút, bỏ qua.
            if abs(center_x - mx) < w and abs(center_y - my) < h:
                is_new = False
                break
                
        if is_new:
            matches.append((center_x, center_y, w, h))
            
    return matches


def human_click(base_x, base_y, template_w, template_h, action_name="Click"):
    offset_x = random.randint(-template_w // 10, template_w // 10)
    offset_y = random.randint(-template_h // 10, template_h // 10)
    
    final_x = base_x + offset_x
    final_y = base_y + offset_y
    
    pyautogui.moveTo(final_x, final_y, duration=random.uniform(0.1, 0.3))
    pyautogui.mouseDown()
    time.sleep(random.uniform(0.05, 0.15))
    pyautogui.mouseUp()
    
    print(f"[{action_name}] Đã click tại ({final_x}, {final_y})")
    
# Tính năng dành cho nâng cao hơn
def check_dice_history(current_screen):
    """
    Quét lịch sử xúc xắc và quyết định xem có nên xài xúc xắc đặc biệt không.
    Trả về con số cần chọn (1, 2, hoặc 3), hoặc trả về None nếu không thỏa mãn.
    """
    # 1. Tạo ROI (Region of Interest) - Hiện tại đang quét chiều dài từ 70%-90%, chiều rộng từ 50%
    # Điều này cực kỳ quan trọng để bot không nhìn nhầm số 1, 2, 3 ở máu hay level.
    h, w = current_screen.shape[:2]
    roi = current_screen[int(h * 0.7):int(h*0.9), int(w*0.5):w] # Cắt lấy phần đáy

    img_rgb = cv2.cvtColor(roi, cv2.COLOR_BGRA2RGB)

    plt.figure(figsize=(10, 6))
    plt.imshow(img_rgb)
    plt.title("Những gì Bot đang nhìn thấy")
    plt.axis('off')
    plt.show()
    
    found_numbers = []
    
    # 2. Quét tìm tất cả các số 1, 2, 3 xuất hiện trong ROI
    for num, img_path in [(0, HIST_NONE), (1, HIST_1), (2, HIST_2), (3, HIST_3)]:
        matches = find_all_image_coordinates(img_path, roi, threshold=0.9) 
        
        for (x, y, w_box, h_box) in matches:
            found_numbers.append({'x': x, 'val': num})

    # Nếu trên màn hình chưa có đủ 2 số (ví dụ mới vào sảnh, chưa đổ lần nào)
    if len(found_numbers) < 2:
        return None

    # 3. MẤU CHỐT: Sắp xếp các số tìm được theo tọa độ X (từ trái qua phải)
    # Ví dụ: [{'x': 120, 'val': 2}, {'x': 180, 'val': 1}] -> Số 2 nằm trước số 1
    found_numbers = sorted(found_numbers, key=lambda item: item['x'])
    
    # Lấy ra 2 số gần nhất (nằm bên phải cùng)
    newest = found_numbers[-1]['val']
    middle = found_numbers[-2]['val']
        
    # 4. Áp dụng Logic của cậu
    if len(found_numbers) >= 3:
        oldest = found_numbers[-3]['val']
        
        # Case lý tưởng: [Cũ] khác [Giữa], và [Giữa] giống [Mới] (VD: 3 1 1)
        if middle == newest and oldest != middle:
            return newest 
        else:
            # Rơi vào case lỡ liên tiếp (1 1 1) hoặc lộn xộn (1 2 1) -> Không làm gì cả
            return None 
    else:
        # Trường hợp mới đổ được 2 lần đầu tiên, chỉ có 2 số hiển thị
        if middle == newest:
            return newest # (VD: _ 1 1)
            
    return None

In [13]:
# DÙNG ĐỂ TEST LẠI TỌA ĐỘ
# Chụp lại màn hình mới nhất
screen_test = np.array(sct.grab(sct.monitors[1]))

dice_coords = find_image_coordinates(STATE_GIFT, screen_test)

if dice_coords:
    print("Tuyệt vời! Đã tìm thấy. Tọa độ:", dice_coords)
else:
    print("Chưa thấy. Thử kiểm tra lại file ảnh hoặc độ phân giải game.")

Chưa thấy. Thử kiểm tra lại file ảnh hoặc độ phân giải game.


In [14]:
# BẮT ĐẦU KHỞI ĐỘNG BOT (BẤM NÚT STOP ĐỂ DỪNG)
print("Bot đang chạy... Bấm nút STOP (⏹) trên Jupyter để dừng.")
time.sleep(2) # Thời gian để cậu alt-tab sang game

# Khởi tạo cờ để chỉ update FIX_COORDS 1 lần duy nhất
is_first_time = True

try:
    while True:
        # 1. Chụp màn hình liên tục
        current_screen = np.array(sct.grab(sct.monitors[1]))
        
        # Check xem đã hoàn thành trận chưa (tìm nút Victory/Skip)
        reward_coords = find_image_coordinates(REWARD_IMG, current_screen)
        if reward_coords:
            print("->  Click lần nhận thưởng!")
            human_click(*FIX_COORDS, action_name="REWARD_CLICK")
            
            time.sleep(2)
            continue # Kết thúc vòng lặp hiện tại, chụp ảnh mới và check lại từ đầu

        victory_coords = find_image_coordinates(VICTORY_IMG, current_screen)
        if victory_coords:
            print("Click lần 2 ra sảnh")
            human_click(*FIX_COORDS, action_name="VICTORY_CLICK")
        
            time.sleep(2)
            continue

        
        # Check có boss x3 không, nếu có thì nhót
        if find_image_coordinates(STATE_BOSS, current_screen) and find_image_coordinates(EXTRA_STATE, current_screen):
            print("-> Đã tới boss cuối. Thoát thôi")
            return_coords = find_image_coordinates(BTN_RETURN, current_screen)
            human_click(*return_coords, action_name="QUIT")
            
            time.sleep(2)
            continue
        
        # Kiểm tra có nút challenge
        challenge_coords = find_image_coordinates(BTN_CHALLENGE, current_screen)
        print(challenge_coords)
        
        if challenge_coords:
            print("-> Thấy quái! Đang bấm Challenge...")
            human_click(*challenge_coords, action_name="CHALLENGE")
            
            # Bấm xong thì game load vào trận, nghỉ lâu một chút
            time.sleep(10)
            continue
        
        # Check xem có phải Quiz
        if find_image_coordinates(STATE_QUIZ, current_screen):
            quiz_coords = find_image_coordinates(STATE_QUIZ, current_screen)
            print("-> [STATE: QUIZ] Phát hiện Fun Quiz!")
            
            answers_list = find_all_image_coordinates(BTN_QUIZ_CORNER, current_screen)
            if len(answers_list) > 0:
                print(f"-> Đã quét thấy {len(answers_list)} đáp án. Đang random chọn đại...")
                
                # Bốc ngẫu nhiên 1 tọa độ trong list để click
                chosen_answer = random.choice(answers_list)
                
                click_x = chosen_answer[0]
                click_y = chosen_answer[1]
                
                human_click(click_x, click_y, 50, 20, action_name="QUIZ_ANSWER")
            else:
                print("-> Quái lạ, không quét thấy nút đáp án nào! (Kiểm tra lại ảnh mẫu góc viền)")
            
            time.sleep(2)
            continue
        
        # Check xem có phải Roll Game
        if find_image_coordinates(STATE_ROLL, current_screen):
            roll_coords = find_image_coordinates(STATE_ROLL, current_screen)
            print("-> [STATE: QUIZ] Phát hiện Dice Game!")
            human_click(*roll_coords, action_name="ROLL_DICE")
            
            time.sleep(2)
            continue
        
        # Check xem có phải Gift
        if find_image_coordinates(STATE_GIFT, current_screen):
            close_coords = find_image_coordinates(BTN_GCLOSE, current_screen)
            print("-> [STATE: QUIZ] Phát hiện GIFT. Đóng con mẹ nó lại")
            human_click(*close_coords, action_name="CLOSE_GIFT")
            
            # Dừng 1s để đợi game hiện cái pop-up "Are you sure..."
            time.sleep(1)
                
            # --- KHÚC QUAN TRỌNG NHẤT ---
            # Phải bắt bot chụp lại màn hình lúc này!
            new_screen = np.array(sct.grab(sct.monitors[1])) 
            
            # Tìm nút OK trên màn hình MỚI
            confirm_coords = find_image_coordinates(BTN_GCONFIRM, new_screen)
            
            if confirm_coords:
                print("-> Xác nhận tắt phần thưởng (Bấm OK)")
                human_click(*confirm_coords, action_name="Confirm")
            else:
                print("-> Quái lạ, không thấy nút OK đâu...")
            
            time.sleep(1)
            
            continue
        
        # Check xem có ở sảnh để roll xúc xắc không
        dice_coords = find_image_coordinates(BTN_DICE, current_screen)
        
        if dice_coords:
            print("-> Thấy Xúc xắc! Đang đổ...")
            human_click(*dice_coords, action_name="ĐỔ XÚC XẮC")
            
            if is_first_time:
                FIX_COORDS = dice_coords
                is_first_time = False
            
            # Sau khi đổ, nhân vật cần thời gian chạy trên bàn cờ. 
            # Nghỉ khoảng 1-2 giây chờ xem có gặp quái không.
            time.sleep(random.uniform(1, 2))
            continue 
            
        # 4. Trạng thái chờ: Không thấy chi hết
        time.sleep(1) # Nghỉ 1s rồi quét lại để đỡ tốn CPU
        
except KeyboardInterrupt:
    print("\n[BOT ĐÃ DỪNG] Cậu vừa bấm ngắt (Interrupt).")

Bot đang chạy... Bấm nút STOP (⏹) trên Jupyter để dừng.
None

[BOT ĐÃ DỪNG] Cậu vừa bấm ngắt (Interrupt).
